In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# load metric.csv
df = pd.read_csv("metric.csv")

# parse timestamp to UTC datetimes (always keep original 'timestamp' column intact)
df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# compute global statistics per (cmdb_id, kpi_name) using the full series (before filtering)
grp_full = df.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    total_count=('value', 'count'),
    global_p95=('value', lambda s: float(s.quantile(0.95)))
)

# define incident window (UTC, inclusive)
start = pd.Timestamp('2024-01-20 10:21:19', tz='UTC')
end   = pd.Timestamp('2024-01-20 10:51:19', tz='UTC')

# filter df to incident window (inclusive)
df_incident = df[(df['ts'] >= start) & (df['ts'] <= end)]

# compute incident stats per pair
grp_inc = df_incident.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    incident_count=('value', 'count'),
    incident_first_ts=('ts', 'min'),
    incident_last_ts=('ts', 'max')
)

# merge full stats with incident stats (keep all pairs)
merged = grp_full.merge(grp_inc, on=['cmdb_id', 'kpi_name'], how='left')

# fill missing incident_count with 0 and leave timestamps as NaT
merged['incident_count'] = merged['incident_count'].fillna(0).astype(int)
# incident_first_ts and incident_last_ts remain NaT where absent

# prepare final table sorted and limited to top 50 rows
result_df = merged.sort_values(by=['incident_count', 'global_p95'], ascending=[False, False]) \
                  .loc[:, ['cmdb_id', 'kpi_name', 'total_count', 'global_p95', 'incident_count', 'incident_first_ts', 'incident_last_ts']] \
                  .head(50) \
                  .reset_index(drop=True)

# unique lists from the metric file (full file)
unique_cmdb = pd.Series(df['cmdb_id'].unique(), name='unique_cmdb')
unique_kpi = pd.Series(df['kpi_name'].unique(), name='unique_kpi_name')

# display the results (as variables)
result_df, unique_cmdb, unique_kpi

```
Out[1]:
```
```python
# Summarize results from previous analysis (variables result_df, unique_cmdb, unique_kpi exist in the kernel)

# prepare counts
num_cmdb = int(len(unique_cmdb))
num_kpi = int(len(unique_kpi))

# incident window info
window_start = "2024-01-20T10:21:19Z"
window_end   = "2024-01-20T10:51:19Z"

# build top 5 summary lines from result_df (already sorted by incident_count desc, global_p95 desc)
top5 = result_df.head(5)
lines = []
lines.append(f"Incident window (UTC): {window_start} to {window_end}")
lines.append(f"Unique cmdb_id count: {num_cmdb}; Unique kpi_name count: {num_kpi}")
lines.append("")
lines.append("Top 5 (by incident_count then global_p95):")
for i, row in enumerate(top5.itertuples(index=False), start=1):
    cmdb, kpi, total_count, global_p95, incident_count, first_ts, last_ts = row
    first_s = first_ts.isoformat() if pd.notna(first_ts) else "null"
    last_s  = last_ts.isoformat()  if pd.notna(last_ts)  else "null"
    lines.append(f"{i}. {cmdb} / {kpi} — incident_count={incident_count}, total_count={total_count}, global_p95={global_p95:.6g}, first={first_s}, last={last_s}")

lines.append("")
lines.append("Note: Pairs with incident_count=0 are included in the full table but not shown above.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id  kpi_name  total_count    global_p95  incident_count         incident_first_ts          incident_last_ts
0   catalogue-db       mem           25  4.305465e+08              25 2024-01-20 10:24:00+00:00 2024-01-20 10:48:00+00:00
1         orders       mem           25  3.258813e+08              25 2024-01-20 10:24:00+00:00 2024-01-20 10:48:00+00:00
2       shipping       mem           25  3.026542e+08              25 2024-01-20 10:24:00+00:00 2024-01-20 10:48:00+00:00
3   queue-master       mem           25  2.559406e+08              25 2024-01-20 10:24:00+00:00 2024-01-20 10:48:00+00:00
4          carts       mem           25  2.144071e+08              25 2024-01-20 10:24:00+00:00 2024-01-20 10:48:00+00:00
..           ...       ...          ...           ...             ...                       ...                       ...
45        orders  workload           25  2.063130e+00              25 2024-01-20 10:24:00+00:00 2024-01-20 10:48:00+00:00
46         carts       cpu           25  1.597313e+00              25 2024-01-20 10:24:00+00:00 2024-01-20 10:48:00+00:00
47       user-db       cpu           25  1.171513e+00              25 2024-01-20 10:24:00+00:00 2024-01-20 10:48:00+00:00
48          user       cpu           25  1.068086e+00              25 2024-01-20 10:24:00+00:00 2024-01-20 10:48:00+00:00
49      shipping       cpu           25  1.006392e+00              25 2024-01-20 10:24:00+00:00 2024-01-20 10:48:00+00:00

[50 rows x 7 columns], 0                 carts
1              carts-db
2             catalogue
3          catalogue-db
4             front-end
5                orders
6             orders-db
7               payment
8          queue-master
9              rabbitmq
10    rabbitmq-exporter
11           session-db
12             shipping
13                 user
14              user-db
Name: unique_cmdb, dtype: object, 0           cpu
1    latency-50
2    latency-90
3           mem
4        socket
5      workload
6        diskio
7         error
Name: unique_kpi_name, dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse existing variables: df, grp_full, df_incident (if any). If not present, recreate them.
import pandas as pd
import numpy as np

# Ensure df is loaded
try:
    df
except NameError:
    df = pd.read_csv("metric.csv")
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Ensure grp_full (global p95 per pair) exists
if 'grp_full' not in globals():
    grp_full = df.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        total_count=('value', 'count'),
        global_p95=('value', lambda s: float(s.quantile(0.95)))
    )

# Incident window (UTC, inclusive)
start = pd.Timestamp('2024-01-20 10:21:19', tz='UTC')
end   = pd.Timestamp('2024-01-20 10:51:19', tz='UTC')

# Ensure df['ts'] exists
if 'ts' not in df.columns:
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Filter to incident window
df_incident = df[(df['ts'] >= start) & (df['ts'] <= end)].copy()

# Merge incident rows with global_p95 so each row has its pair's global_p95
if df_incident.shape[0] > 0:
    inc_merged = df_incident.merge(grp_full[['cmdb_id', 'kpi_name', 'global_p95']],
                                   on=['cmdb_id', 'kpi_name'],
                                   how='left')
    # mark anomalies where value > global_p95
    inc_merged['is_anom'] = inc_merged['value'] > inc_merged['global_p95']
    # groupby to compute required incident-window metrics
    agg_inc = inc_merged.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        anomaly_count=('is_anom', 'sum'),
        # earliest ts where is_anom is True; compute via conditional
        anomaly_first_ts=('ts', lambda s: s[inc_merged.loc[s.index, 'is_anom']].min() if len(s)>0 else pd.NaT),
        anomaly_max_value=('value', 'max')
    )
else:
    # No incident rows at all: create empty agg_inc with correct columns
    agg_inc = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','anomaly_first_ts','anomaly_max_value'])

# Start from full set of pairs (grp_full) and left-merge agg_inc so pairs with no incident rows are included
anom_df = grp_full[['cmdb_id', 'kpi_name', 'global_p95']].merge(agg_inc, on=['cmdb_id', 'kpi_name'], how='left')

# Fill missing anomaly_count with 0 and keep timestamps as NaT; anomaly_max_value may be NaN if no incident rows
anom_df['anomaly_count'] = anom_df['anomaly_count'].fillna(0).astype(int)

# anomaly_first_ts: already NaT where missing; ensure dtype is datetime64[ns, UTC]
if 'anomaly_first_ts' in anom_df.columns:
    anom_df['anomaly_first_ts'] = pd.to_datetime(anom_df['anomaly_first_ts'], utc=True)

# Compute percent_over: (anomaly_max_value - global_p95)/global_p95, but if global_p95==0 -> set NaN
anom_df['percent_over'] = np.where(
    (anom_df['global_p95'] == 0) | (anom_df['anomaly_max_value'].isna()),
    np.nan,
    (anom_df['anomaly_max_value'] - anom_df['global_p95']) / anom_df['global_p95']
)

# total_anomalous_pairs: count of pairs with anomaly_count > 0
total_anomalous_pairs = int((anom_df['anomaly_count'] > 0).sum())

# Prepare final table sorted by anomaly_count desc then percent_over desc, limit top 20
result_anom = anom_df.sort_values(by=['anomaly_count', 'percent_over'], ascending=[False, False]) \
                     .loc[:, ['cmdb_id', 'kpi_name', 'global_p95', 'anomaly_count', 'anomaly_first_ts', 'anomaly_max_value', 'percent_over']] \
                     .head(20) \
                     .reset_index(drop=True)

# Display the results (DataFrame and scalar)
result_anom, total_anomalous_pairs

```
Out[2]:
```
```python
# Summarize the anomaly analysis results as plain English.
# Reuse existing variables: result_anom (top rows) and total_anomalous_pairs (scalar).

try:
    result_anom
    total_anomalous_pairs
except NameError:
    # safety: reload results if not present (shouldn't be needed)
    import pandas as pd
    df = pd.read_csv("metric.csv")
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    grp_full = df.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        total_count=('value', 'count'),
        global_p95=('value', lambda s: float(s.quantile(0.95)))
    )
    start = pd.Timestamp('2024-01-20 10:21:19', tz='UTC')
    end   = pd.Timestamp('2024-01-20 10:51:19', tz='UTC')
    df_incident = df[(df['ts'] >= start) & (df['ts'] <= end)].copy()
    inc_merged = df_incident.merge(grp_full[['cmdb_id','kpi_name','global_p95']], on=['cmdb_id','kpi_name'], how='left')
    inc_merged['is_anom'] = inc_merged['value'] > inc_merged['global_p95']
    agg_inc = inc_merged.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        anomaly_count=('is_anom','sum'),
        anomaly_first_ts=('ts', lambda s: s[inc_merged.loc[s.index,'is_anom']].min() if len(s)>0 else pd.NaT),
        anomaly_max_value=('value','max')
    )
    result_anom = grp_full[['cmdb_id','kpi_name','global_p95']].merge(agg_inc, on=['cmdb_id','kpi_name'], how='left')
    result_anom['anomaly_count'] = result_anom['anomaly_count'].fillna(0).astype(int)
    result_anom['anomaly_first_ts'] = pd.to_datetime(result_anom['anomaly_first_ts'], utc=True)
    import numpy as np
    result_anom['percent_over'] = np.where(
        (result_anom['global_p95'] == 0) | (result_anom['anomaly_max_value'].isna()),
        np.nan,
        (result_anom['anomaly_max_value'] - result_anom['global_p95']) / result_anom['global_p95']
    )
    total_anomalous_pairs = int((result_anom['anomaly_count'] > 0).sum())
    result_anom = result_anom.sort_values(by=['anomaly_count','percent_over'], ascending=[False, False]).head(20).reset_index(drop=True)

# Build summary string
lines = []
lines.append("Incident window (UTC): 2024-01-20 10:21:19 to 2024-01-20 10:51:19")
lines.append(f"Total (cmdb_id, kpi_name) pairs with anomalies in window: {int(total_anomalous_pairs)}")
lines.append("")
lines.append("Top anomalous pairs (sorted by anomaly_count then percent_over) — top 5:")
top5 = result_anom.head(5)
if top5.shape[0] == 0:
    lines.append("  No anomalies detected in the incident window.")
else:
    for i, r in enumerate(top5.itertuples(index=False), start=1):
        cmdb = r.cmdb_id
        kpi = r.kpi_name
        an_count = int(r.anomaly_count)
        first = r.anomaly_first_ts.isoformat() if pd.notna(r.anomaly_first_ts) else "null"
        maxv = float(r.anomaly_max_value) if (hasattr(r, 'anomaly_max_value') and pd.notna(r.anomaly_max_value)) else None
        p = float(r.percent_over) if (hasattr(r, 'percent_over') and pd.notna(r.percent_over)) else None
        if p is None:
            p_str = "null"
        else:
            p_str = f"{p*100:.1f}%"
        maxv_str = f"{maxv:.6g}" if maxv is not None else "null"
        lines.append(f"  {i}. {cmdb} / {kpi} — anomaly_count={an_count}, first={first}, anomaly_max_value={maxv_str}, percent_over={p_str}")

lines.append("")
lines.append("Interpretation:")
lines.append("  - There are multiple metric pairs that exceeded their historical (global) 95th percentile during the incident window.")
lines.append("  - The most extreme relative exceedance in the top results is orders/diskio (anomaly_max_value ~1.59e5, ~122.9% above its global_p95).")
lines.append("  - Use these top pairs as starting points for root-cause investigation (disk IO and CPU anomalies are prominent).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name    global_p95  anomaly_count          anomaly_first_ts  anomaly_max_value  percent_over
0              orders      diskio  7.154601e+04              2 2024-01-20 10:24:00+00:00       1.594971e+05      1.229294
1          session-db      diskio  1.594327e+04              2 2024-01-20 10:37:00+00:00       2.895680e+04      0.816239
2        queue-master         cpu  9.507267e-01              2 2024-01-20 10:38:00+00:00       1.153437e+00      0.213216
3            shipping         cpu  1.006392e+00              2 2024-01-20 10:33:00+00:00       1.210607e+00      0.202918
4             payment  latency-90  5.039027e-03              2 2024-01-20 10:36:00+00:00       6.044618e-03      0.199561
5           front-end      socket  1.196667e+01              2 2024-01-20 10:41:00+00:00       1.361667e+01      0.137883
6   rabbitmq-exporter         cpu  1.453193e-04              2 2024-01-20 10:40:00+00:00       1.601782e-04      0.102250
7              orders         cpu  3.240340e+00              2 2024-01-20 10:37:00+00:00       3.566744e+00      0.100732
8              orders  latency-90  3.441930e+00              2 2024-01-20 10:41:00+00:00       3.770417e+00      0.095437
9            shipping      socket  6.250000e+00              2 2024-01-20 10:27:00+00:00       6.833333e+00      0.093333
10          catalogue    workload  4.194680e+00              2 2024-01-20 10:31:00+00:00       4.435550e+00      0.057423
11           shipping  latency-90  5.626191e-03              2 2024-01-20 10:36:00+00:00       5.924561e-03      0.053032
12       queue-master      diskio  4.627521e+04              2 2024-01-20 10:26:00+00:00       4.797663e+04      0.036767
13           rabbitmq         cpu  8.594851e-01              2 2024-01-20 10:39:00+00:00       8.877341e-01      0.032867
14           shipping  latency-50  3.165861e-03              2 2024-01-20 10:36:00+00:00       3.262910e-03      0.030655
15            user-db      diskio  1.599077e+06              2 2024-01-20 10:32:00+00:00       1.644420e+06      0.028356
16             orders    workload  2.063130e+00              2 2024-01-20 10:32:00+00:00       2.117950e+00      0.026571
17           carts-db      diskio  2.349733e+06              2 2024-01-20 10:42:00+00:00       2.411963e+06      0.026484
18             orders  latency-50  6.982982e-01              2 2024-01-20 10:42:00+00:00       7.162739e-01      0.025742
19             orders      socket  2.150000e+01              2 2024-01-20 10:43:00+00:00       2.201667e+01      0.024031, 62)```
```

In [ ]:
# In[3]:

# Recompute consecutive anomaly runs with fixed grouping assignment (avoid index-mismatch error).
import pandas as pd
import numpy as np

# Ensure data and global p95 exist
try:
    df
except NameError:
    df = pd.read_csv("metric.csv")
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

if 'grp_full' not in globals():
    grp_full = df.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        total_count=('value', 'count'),
        global_p95=('value', lambda s: float(s.quantile(0.95)))
    )

# Incident window
start = pd.Timestamp('2024-01-20 10:21:19', tz='UTC')
end   = pd.Timestamp('2024-01-20 10:51:19', tz='UTC')

df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
df_incident = df[(df['ts'] >= start) & (df['ts'] <= end)].copy()

if df_incident.empty:
    runs_df = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes','run_max_value','run_percent_over','run_significant'])
    significant_cmdb_count = 0
else:
    inc_merged = df_incident.merge(grp_full[['cmdb_id','kpi_name','global_p95']],
                                   on=['cmdb_id','kpi_name'], how='left')
    inc_merged = inc_merged.sort_values(['cmdb_id','kpi_name','ts']).reset_index(drop=True)
    inc_merged['is_anom'] = inc_merged['value'] > inc_merged['global_p95']

    # Build segments per group safely by iterating groups and concatenating
    parts = []
    for (cmdb, kpi), g in inc_merged.groupby(['cmdb_id','kpi_name'], sort=False):
        g = g.copy()
        # segment id increments whenever is_anom changes (including from False->True)
        g['seg'] = (g['is_anom'] != g['is_anom'].shift(fill_value=False)).cumsum()
        parts.append(g)
    inc_with_seg = pd.concat(parts, ignore_index=True)

    # Keep only anomaly segments (is_anom == True)
    anom_rows = inc_with_seg[inc_with_seg['is_anom']].copy()
    if anom_rows.empty:
        runs_df = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes','run_max_value','run_percent_over','run_significant'])
        significant_cmdb_count = 0
    else:
        runs = anom_rows.groupby(['cmdb_id','kpi_name','seg'], as_index=False).agg(
            run_start_ts = ('ts', 'min'),
            run_end_ts   = ('ts', 'max'),
            run_length_minutes = ('ts', 'size'),
            run_max_value = ('value', 'max'),
            global_p95 = ('global_p95', 'first')
        )
        runs['run_percent_over'] = np.where(
            runs['global_p95'] == 0,
            np.nan,
            (runs['run_max_value'] - runs['global_p95']) / runs['global_p95']
        )
        runs['run_significant'] = runs['run_percent_over'] > 0.5
        # count distinct cmdb_id with at least one significant run (use all runs)
        significant_cmdb_count = int(runs[runs['run_significant']]['cmdb_id'].nunique())
        runs_df = runs.loc[:, ['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes','run_max_value','run_percent_over','run_significant']]
        runs_df = runs_df.sort_values(by=['run_length_minutes','run_percent_over'], ascending=[False, False]).head(20).reset_index(drop=True)

# Return the runs (up to 20) and the scalar count of distinct cmdb_id with significant runs
runs_df, significant_cmdb_count

```
Out[3]:
```
```python
# Summarize the run analysis using existing variables runs_df and significant_cmdb_count (from previous execution).
try:
    runs_df
    significant_cmdb_count
except NameError:
    raise RuntimeError("Required variables 'runs_df' and 'significant_cmdb_count' are not available in the kernel.")

# Compute counts from the returned runs (runs_df contains up to top 20 runs)
total_runs_returned = int(len(runs_df))
significant_runs_count = int(runs_df['run_significant'].sum())

# Build top 3 run summaries
top3 = []
for r in runs_df.head(3).itertuples(index=False):
    cmdb = r.cmdb_id
    kpi = r.kpi_name
    start = r.run_start_ts.isoformat()
    end = r.run_end_ts.isoformat()
    length = int(r.run_length_minutes)
    maxv = float(r.run_max_value)
    p = float(r.run_percent_over) if pd.notna(r.run_percent_over) else None
    p_str = f"{p*100:.1f}%" if p is not None else "null"
    sig = bool(r.run_significant)
    top3.append(f"{cmdb}/{kpi}: {length} min, {start} to {end}, max={maxv:.6g}, over={p_str}, significant={sig}")

summary_lines = [
    "Incident window: 2024-01-20 10:21:19 UTC to 2024-01-20 10:51:19 UTC",
    f"Total runs returned (top 20 sorted): {total_runs_returned}",
    f"Runs marked significant (run_percent_over > 0.5) in the returned set: {significant_runs_count}",
    f"Distinct cmdb_id with at least one significant run in the window: {int(significant_cmdb_count)}",
    "",
    "Top 3 runs (by length then percent_over):"
]
summary_lines += [f"  {i+1}. {t}" for i, t in enumerate(top3)]
summary_lines += [
    "",
    "Notes:",
    "  - The largest and most significant run is orders/diskio (2 minutes, run_percent_over ≈ 122.9%),",
    "    indicating a large disk IO spike on 'orders' during the incident window.",
    "  - session-db/diskio also had a significant single-minute run (≈ 81.6% over its global p95).",
    "  - Use these services/components (orders and session-db disk IO) as primary leads for root-cause investigation."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name              run_start_ts                run_end_ts  run_length_minutes  run_max_value  run_percent_over  run_significant
0         orders      diskio 2024-01-20 10:24:00+00:00 2024-01-20 10:25:00+00:00                   2   1.594971e+05          1.229294             True
1       shipping         cpu 2024-01-20 10:33:00+00:00 2024-01-20 10:34:00+00:00                   2   1.210607e+00          0.202918            False
2        payment  latency-90 2024-01-20 10:36:00+00:00 2024-01-20 10:37:00+00:00                   2   6.044618e-03          0.199561            False
3      front-end      socket 2024-01-20 10:41:00+00:00 2024-01-20 10:42:00+00:00                   2   1.361667e+01          0.137883            False
4       shipping  latency-90 2024-01-20 10:36:00+00:00 2024-01-20 10:37:00+00:00                   2   5.924561e-03          0.053032            False
5       shipping  latency-50 2024-01-20 10:36:00+00:00 2024-01-20 10:37:00+00:00                   2   3.262910e-03          0.030655            False
6         orders    workload 2024-01-20 10:32:00+00:00 2024-01-20 10:33:00+00:00                   2   2.117950e+00          0.026571            False
7   catalogue-db         cpu 2024-01-20 10:42:00+00:00 2024-01-20 10:43:00+00:00                   2   2.105514e-01          0.020600            False
8      front-end  latency-50 2024-01-20 10:34:00+00:00 2024-01-20 10:35:00+00:00                   2   3.299412e-02          0.008654            False
9        user-db         mem 2024-01-20 10:47:00+00:00 2024-01-20 10:48:00+00:00                   2   5.314601e+07          0.007741            False
10     catalogue         mem 2024-01-20 10:47:00+00:00 2024-01-20 10:48:00+00:00                   2   6.265446e+06          0.001810            False
11      carts-db         mem 2024-01-20 10:47:00+00:00 2024-01-20 10:48:00+00:00                   2   8.699802e+07          0.001286            False
12     orders-db         mem 2024-01-20 10:47:00+00:00 2024-01-20 10:48:00+00:00                   2   4.359864e+07          0.000828            False
13        orders         mem 2024-01-20 10:44:00+00:00 2024-01-20 10:45:00+00:00                   2   3.261034e+08          0.000682            False
14         carts         mem 2024-01-20 10:47:00+00:00 2024-01-20 10:48:00+00:00                   2   2.144205e+08          0.000062            False
15      rabbitmq         mem 2024-01-20 10:34:00+00:00 2024-01-20 10:35:00+00:00                   2   9.595672e+07          0.000050            False
16    session-db      diskio 2024-01-20 10:48:00+00:00 2024-01-20 10:48:00+00:00                   1   2.895680e+04          0.816239             True
17     front-end       error 2024-01-20 10:31:00+00:00 2024-01-20 10:31:00+00:00                   1   6.022833e-01          0.270921            False
18  queue-master         cpu 2024-01-20 10:38:00+00:00 2024-01-20 10:38:00+00:00                   1   1.153437e+00          0.213216            False
19        orders       error 2024-01-20 10:31:00+00:00 2024-01-20 10:31:00+00:00                   1   2.999167e-01          0.209179            False, 2)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Load trace.csv and parse timestamps to UTC datetimes
df_trace = pd.read_csv("trace.csv")
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Candidate services and incident window (UTC)
candidates = ["carts","catalogue","orders","payment","user"]
start = pd.Timestamp('2024-01-20 10:21:19', tz='UTC')
end   = pd.Timestamp('2024-01-20 10:51:19', tz='UTC')

# Compute full-series stats (total_count and global_p95) per (cmdb_id, trace_name)
grp_full_trace = df_trace.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    total_count=('value', 'count'),
    global_p95=('value', lambda s: float(s.quantile(0.95)))
)

# Restrict to candidate services for incident analysis
grp_full_trace_cand = grp_full_trace[grp_full_trace['cmdb_id'].isin(candidates)].copy()

# Unique trace_name values present for the candidate services (from full series)
unique_traces = pd.Series(sorted(df_trace[df_trace['cmdb_id'].isin(candidates)]['trace_name'].unique()), name='unique_trace_name')

# Filter trace points to incident window and candidate services
df_trace_incident = df_trace[
    (df_trace['ts'] >= start) &
    (df_trace['ts'] <= end) &
    (df_trace['cmdb_id'].isin(candidates))
].copy()

if not df_trace_incident.empty:
    # Merge incident rows with global_p95 so each incident row has its pair's global_p95
    inc_merged = df_trace_incident.merge(
        grp_full_trace_cand[['cmdb_id', 'trace_name', 'global_p95']],
        on=['cmdb_id', 'trace_name'],
        how='left'
    )
    # mark anomalies where value > global_p95
    inc_merged['is_anom'] = inc_merged['value'] > inc_merged['global_p95']
    inc_merged = inc_merged.sort_values(['cmdb_id', 'trace_name', 'ts']).reset_index(drop=True)

    # Aggregate incident-window metrics per pair
    agg_inc = inc_merged.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
        incident_count=('value', 'count'),
        anomaly_count=('is_anom', 'sum'),
        # earliest ts where is_anom is True (or NaT)
        anomaly_first_ts=('ts', lambda s: s[inc_merged.loc[s.index, 'is_anom']].min() if len(s) > 0 else pd.NaT),
        anomaly_max_value=('value', 'max')
    )
else:
    # No incident rows for candidates
    agg_inc = pd.DataFrame(columns=['cmdb_id','trace_name','incident_count','anomaly_count','anomaly_first_ts','anomaly_max_value'])

# Merge full-series stats (for candidates) with incident aggregates (left join to keep all pairs)
result_trace = grp_full_trace_cand.merge(agg_inc, on=['cmdb_id', 'trace_name'], how='left')

# Fill missing counts with 0; keep timestamps as NaT
result_trace['incident_count'] = result_trace['incident_count'].fillna(0).astype(int)
result_trace['anomaly_count'] = result_trace['anomaly_count'].fillna(0).astype(int)
result_trace['anomaly_first_ts'] = pd.to_datetime(result_trace['anomaly_first_ts'], utc=True)

# Compute percent_over using global_p95 (null if global_p95 == 0 or anomaly_max_value is missing)
result_trace['anomaly_max_value'] = result_trace['anomaly_max_value'].astype(float)
result_trace['percent_over'] = np.where(
    (result_trace['global_p95'] == 0) | (result_trace['anomaly_max_value'].isna()),
    np.nan,
    (result_trace['anomaly_max_value'] - result_trace['global_p95']) / result_trace['global_p95']
)

# Select required columns and sort by anomaly_count desc then percent_over desc, limit top 20
result_trace = result_trace.sort_values(by=['anomaly_count', 'percent_over'], ascending=[False, False]) \
    .loc[:, ['cmdb_id', 'trace_name', 'total_count', 'global_p95', 'incident_count', 'anomaly_count', 'anomaly_first_ts', 'anomaly_max_value', 'percent_over']] \
    .head(20) \
    .reset_index(drop=True)

# Display results (DataFrame and the list of distinct trace_name values for candidate services)
result_trace, unique_traces

```
Out[4]:
```
```python
# Summary of trace analysis for candidate services during the incident window
summary = (
    "Incident window (UTC): 2024-01-20 10:21:19 — 2024-01-20 10:51:19.\n"
    "Result: No trace rows found for the candidate services ['carts', 'catalogue', 'orders', 'payment', 'user'] "
    "within the incident window. The result table is empty and there are no trace_name values present for these services.\n"
    "Interpretation: No trace-based anomalies could be detected for the specified candidate services in that time window. "
    "Please verify trace ingestion or expand the candidate set / window if further investigation is needed."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, trace_name, total_count, global_p95, incident_count, anomaly_count, anomaly_first_ts, anomaly_max_value, percent_over]
Index: [], Series([], Name: unique_trace_name, dtype: object))```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Candidate services and incident window (UTC)
candidates = ["carts","catalogue","orders","payment","user"]
start = pd.Timestamp('2024-01-20 10:21:19', tz='UTC')
end   = pd.Timestamp('2024-01-20 10:51:19', tz='UTC')

# Load log.csv and parse timestamps
df_log = pd.read_csv("log.csv")
df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Compute full-series stats per (cmdb_id, log_name): total_count and global_p95 (do NOT use incident-only data)
grp_full_log = df_log.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
    total_count=('value', 'count'),
    global_p95=('value', lambda s: float(s.quantile(0.95)))
)

# Filter log points to incident window and candidate services
df_log_incident = df_log[
    (df_log['ts'] >= start) &
    (df_log['ts'] <= end) &
    (df_log['cmdb_id'].isin(candidates))
].copy()

if not df_log_incident.empty:
    # Merge incident rows with global_p95 so each incident row has its pair's global_p95
    inc_merged = df_log_incident.merge(
        grp_full_log[['cmdb_id', 'log_name', 'global_p95']],
        on=['cmdb_id', 'log_name'],
        how='left'
    )
    inc_merged = inc_merged.sort_values(['cmdb_id', 'log_name', 'ts']).reset_index(drop=True)
    inc_merged['is_anom'] = inc_merged['value'] > inc_merged['global_p95']

    # Aggregate incident-window metrics per pair
    agg_inc = inc_merged.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
        incident_count=('value', 'count'),
        anomaly_count=('is_anom', 'sum'),
        anomaly_first_ts=('ts', lambda s: s[inc_merged.loc[s.index, 'is_anom']].min() if inc_merged.loc[s.index, 'is_anom'].any() else pd.NaT),
        anomaly_max_value=('value', 'max')
    )
else:
    agg_inc = pd.DataFrame(columns=['cmdb_id','log_name','incident_count','anomaly_count','anomaly_first_ts','anomaly_max_value'])

# Merge full-series stats with incident aggregates (keep all pairs from full series but we will restrict to candidates later)
result_log = grp_full_log.merge(agg_inc, on=['cmdb_id', 'log_name'], how='left')

# Restrict to candidate services for final output
result_log = result_log[result_log['cmdb_id'].isin(candidates)].copy()

# Fill missing incident/anomaly counts with 0; timestamps remain NaT
result_log['incident_count'] = result_log['incident_count'].fillna(0).astype(int)
result_log['anomaly_count'] = result_log['anomaly_count'].fillna(0).astype(int)
result_log['anomaly_first_ts'] = pd.to_datetime(result_log['anomaly_first_ts'], utc=True)

# Ensure anomaly_max_value is float (NaN if no incident points)
result_log['anomaly_max_value'] = result_log['anomaly_max_value'].astype(float)

# Compute percent_over: (anomaly_max_value - global_p95)/global_p95, null if global_p95 == 0 or anomaly_max_value missing
result_log['percent_over'] = np.where(
    (result_log['global_p95'] == 0) | (result_log['anomaly_max_value'].isna()),
    np.nan,
    (result_log['anomaly_max_value'] - result_log['global_p95']) / result_log['global_p95']
)

# Select required columns and sort by anomaly_count desc then percent_over desc, limit top 20
result_log = result_log.sort_values(by=['anomaly_count', 'percent_over'], ascending=[False, False]) \
    .loc[:, ['cmdb_id', 'log_name', 'total_count', 'global_p95', 'incident_count', 'anomaly_count', 'anomaly_first_ts', 'anomaly_max_value', 'percent_over']] \
    .head(20) \
    .reset_index(drop=True)

# ---- error_logs.csv query ----
df_error = pd.read_csv("error_logs.csv")
df_error['ts'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

df_error_filtered = df_error[
    (df_error['cmdb_id'].isin(candidates)) &
    (df_error['ts'] >= start) &
    (df_error['ts'] <= end)
].copy()

# Prepare final error result: timestamp (UTC datetime), cmdb_id, message; limit to top 50
if df_error_filtered.empty:
    # empty DataFrame with expected columns
    result_errors = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'message'])
else:
    result_errors = df_error_filtered.loc[:, ['ts', 'cmdb_id', 'message']].rename(columns={'ts': 'timestamp'})
    result_errors = result_errors.sort_values('timestamp').head(50).reset_index(drop=True)

# Display compact results
result_log, result_errors

```
Out[5]:
```
```python
# Summarize log and error-log findings in plain English using existing results (result_log, result_errors).
try:
    result_log
    result_errors
except NameError:
    raise RuntimeError("Expected variables 'result_log' and 'result_errors' not found in kernel.")

lines = []
lines.append("Incident window (UTC): 2024-01-20 10:21:19 — 2024-01-20 10:51:19")
lines.append("")
lines.append("Log metric anomalies (top findings):")
if result_log.empty:
    lines.append("  No log metric rows for candidate services in the incident window.")
else:
    # Summarize top 5 rows from result_log (already sorted by anomaly_count, percent_over)
    top = result_log.head(5)
    for i, r in enumerate(top.itertuples(index=False), start=1):
        cmdb = r.cmdb_id
        logn = r.log_name
        an_count = int(r.anomaly_count)
        first_ts = r.anomaly_first_ts.isoformat() if pd.notna(r.anomaly_first_ts) else "null"
        maxv = r.anomaly_max_value
        pct = r.percent_over
        pct_str = f"{pct*100:.1f}%" if pd.notna(pct) else "null"
        lines.append(f"  {i}. {cmdb} / {logn} — anomaly_count={an_count}, first={first_ts}, anomaly_max_value={maxv:.6g}, percent_over={pct_str}")

lines.append("")
lines.append("Notable observations:")
lines.append("  - orders / log.row_count: 2 anomalous points (first at 2024-01-20T10:41:00Z), anomaly_max_value=1404 (~84.3% over its global p95).")
lines.append("  - carts / log.row_count: 1 anomalous point at 2024-01-20T10:41:00Z, anomaly_max_value=200 (100% over its global p95).")
lines.append("  - user and payment also show small row_count exceedances but with low percent_over.")
lines.append("  - log.error_count for these services has global_p95 == 0 and showed no anomalies in the window (anomaly_count=0).")
lines.append("")
lines.append("Error logs:")
if result_errors.empty:
    lines.append("  No error_logs entries found for the candidate services in the incident window.")
else:
    # show up to 5 entries
    for i, r in enumerate(result_errors.head(5).itertuples(index=False), start=1):
        ts = r.timestamp.isoformat()
        lines.append(f"  {i}. {ts} | {r.cmdb_id} | {r.message}")

lines.append("")
lines.append("Interpretation / recommended next steps:")
lines.append("  - The strongest signals are spikes in log.row_count for 'orders' and 'carts' around 10:41 UTC,")
lines.append("    suggesting elevated request/row activity (potential load spike or bursty traffic).")
lines.append("  - No corresponding error log messages were found in error_logs.csv for these services during the window.")
lines.append("  - Investigate traffic/load sources to 'orders' and 'carts' at ~10:41 UTC and check related infrastructure (DB, disk IO, queueing).")
lines.append("  - Correlate these log.row_count spikes with metric and disk IO anomalies already observed (e.g., orders diskio).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id         log_name  total_count  global_p95  incident_count  anomaly_count          anomaly_first_ts  anomaly_max_value  percent_over
0     orders    log.row_count           25       761.6              25              2 2024-01-20 10:41:00+00:00             1404.0      0.843487
1       user    log.row_count           25       610.0              25              2 2024-01-20 10:37:00+00:00              618.0      0.013115
2    payment    log.row_count           25       104.8              25              2 2024-01-20 10:26:00+00:00              106.0      0.011450
3      carts    log.row_count           21       100.0              21              1 2024-01-20 10:41:00+00:00              200.0      1.000000
4  catalogue    log.row_count           25       165.0              25              1 2024-01-20 10:38:00+00:00              166.0      0.006061
5      carts  log.error_count           21         0.0              21              0                       NaT                0.0           NaN
6  catalogue  log.error_count           25         0.0              25              0                       NaT                0.0           NaN
7     orders  log.error_count           25         0.0              25              0                       NaT                0.0           NaN
8    payment  log.error_count           25         0.0              25              0                       NaT                0.0           NaN
9       user  log.error_count           25         0.0              25              0                       NaT                0.0           NaN, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```